# 03 描述性統計與 2×2 表

松柏護理之家退伍軍人症群聚，主管問：「使用淋浴的人，感染風險有沒有比較高？」

這堂課學會：**2×2 列聯表 → 風險比 (RR) → 信賴區間 (CI) → 卡方檢定 → 多因子彙整**。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 資料準備 ---
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from epi_learning.metrics import risk_ratio

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
print(f"全體：{len(df)} 人，感染：{df['infected'].sum()} 人")

In [ ]:
# --- Step 2: 建立 2×2 表（淋浴 × 感染）---
ct_shower = pd.crosstab(
    df["shower_use"], df["infected"],
    margins=True, margins_name="合計",
)
ct_shower.index = ["未使用淋浴", "使用淋浴", "合計"]
ct_shower.columns = ["未感染", "感染", "合計"]
print(ct_shower)

In [ ]:
# --- Step 3: 計算 Risk Ratio ---
a = int(ct_shower.loc["使用淋浴", "感染"])        # 暴露+感染
b = int(ct_shower.loc["使用淋浴", "未感染"])      # 暴露+未感染
c = int(ct_shower.loc["未使用淋浴", "感染"])      # 未暴露+感染
d = int(ct_shower.loc["未使用淋浴", "未感染"])    # 未暴露+未感染

rr = risk_ratio(a, a + b, c, c + d)
print(f"淋浴使用 → 感染的 RR = {rr:.3f}")
print(f"  暴露組風險: {a/(a+b):.1%}")
print(f"  未暴露組風險: {c/(c+d):.1%}")

In [ ]:
# --- Step 4: 95% 信賴區間 ---
# RR 的 CI 用對數轉換法（Katz method）
ln_rr = np.log(rr)
se_ln_rr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))

ci_lower = np.exp(ln_rr - 1.96 * se_ln_rr)
ci_upper = np.exp(ln_rr + 1.96 * se_ln_rr)

print(f"RR = {rr:.3f} (95% CI: {ci_lower:.3f} – {ci_upper:.3f})")

if ci_lower > 1:
    print("→ 95% CI 不包含 1，暴露與感染有統計顯著關聯")
else:
    print("→ 95% CI 包含 1，無法排除暴露與感染無關")

In [ ]:
# --- Step 5: 卡方檢定 ---
contingency = [[a, b], [c, d]]
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"卡方統計量 = {chi2:.3f}")
print(f"自由度 = {dof}")
print(f"p-value = {p:.4f}")
print(f"\n期望值表：")
print(pd.DataFrame(
    expected.round(1),
    index=["使用淋浴", "未使用淋浴"],
    columns=["感染", "未感染"],
))

In [ ]:
# --- Step 6: 第二個暴露因子 — 水療使用 ---
ct_hydro = pd.crosstab(df["hydrotherapy_use"], df["infected"])
a2, b2 = int(ct_hydro.loc[1, 1]), int(ct_hydro.loc[1, 0])
c2, d2 = int(ct_hydro.loc[0, 1]), int(ct_hydro.loc[0, 0])

rr2 = risk_ratio(a2, a2 + b2, c2, c2 + d2)
chi2_2, p2, _, _ = chi2_contingency([[a2, b2], [c2, d2]])

ln_rr2 = np.log(rr2)
se2 = np.sqrt(1/a2 - 1/(a2+b2) + 1/c2 - 1/(c2+d2))
ci2_lo = np.exp(ln_rr2 - 1.96 * se2)
ci2_hi = np.exp(ln_rr2 + 1.96 * se2)

print(f"水療使用 → 感染")
print(f"  RR = {rr2:.3f} (95% CI: {ci2_lo:.3f} – {ci2_hi:.3f})")
print(f"  p-value = {p2:.4f}")

In [ ]:
# --- Step 7: 多因子粗 RR 彙整表 ---
# 一次比較所有可能的危險因子，找出「嫌疑最大」的暴露

factors = [
    "shower_use", "hydrotherapy_use",
    "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed",
]

# smoking_history 是三分類（never/former/current），先轉為二分類
df["ever_smoker"] = (df["smoking_history"] != "never").astype(int)
factors.append("ever_smoker")

results = []
for factor in factors:
    ct = pd.crosstab(df[factor], df["infected"])
    a_i = int(ct.loc[1, 1])
    b_i = int(ct.loc[1, 0])
    c_i = int(ct.loc[0, 1])
    d_i = int(ct.loc[0, 0])
    rr_i = risk_ratio(a_i, a_i + b_i, c_i, c_i + d_i)
    chi2_i, p_i, _, _ = chi2_contingency([[a_i, b_i], [c_i, d_i]])
    ln_rr_i = np.log(rr_i)
    se_i = np.sqrt(1/a_i - 1/(a_i+b_i) + 1/c_i - 1/(c_i+d_i))
    ci_lo = np.exp(ln_rr_i - 1.96 * se_i)
    ci_hi = np.exp(ln_rr_i + 1.96 * se_i)
    results.append({
        "factor": factor,
        "RR": round(rr_i, 3),
        "95% CI lower": round(ci_lo, 3),
        "95% CI upper": round(ci_hi, 3),
        "p-value": round(p_i, 4),
    })

rr_table = pd.DataFrame(results).sort_values("RR", ascending=False)
print("=== 多因子粗 RR 彙整表 ===")
print(rr_table.to_string(index=False))

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 2×2 表 | `pd.crosstab()` |
| RR | `risk_ratio()` + 手動計算 |
| 95% CI | 對數轉換法（Katz method） |
| 卡方檢定 | `scipy.stats.chi2_contingency()` |
| 多因子掃描 | 迴圈 + 彙整成 DataFrame |

**重要提醒**：粗 RR 只是初步線索。淋浴使用的 RR 看起來很高，但可能被**交絡（confounding）**影響——例如高樓層同時淋浴使用率高又靠近水塔。Ch05 會用**分層分析**和 **Mantel-Haenszel 法**來處理這個問題。